# RakshaDoc AI — Training a Layout Network From Scratch

## Read this first: what this notebook is and is not

### It is NOT

- **Not** a reproduction of anything the RakshaDoc AI backend runs. The backend has **no trained weights of its own**.
- **Not** a reimplementation of Deformable DETR. That model is used pretrained; retraining it from scratch is a multi-GPU-days project.
- **Not** a claim about production quality. What you train here will be a working demonstration, not a benchmark-beating detector.

### It IS

A **small, honest, end-to-end object detection network** written from zero, trained on a **synthetic dataset generated in this notebook**, so you can see every moving part: dataset → augmentation → model → loss → training loop → evaluation → export.

### Why synthetic data?

Real document layout datasets (DocLayNet, PubLayNet) require account-gated downloads of many gigabytes. Generating the data keeps the notebook self-contained and runnable in one click, and it makes the ground truth **exact** — no annotation noise to explain away. The techniques transfer; the numbers do not.

### Architecture choice

A tiny **anchor-free, single-scale FCOS-style** detector:

- **Backbone:** 5 downsampling conv blocks (stride 32 feature map)
- **Head:** shared 3-conv tower per cell predicting `(centerness, 4 box offsets, 3 class logits)`
- **Loss:** focal loss for classification + IoU (GIoU) loss for regression, following the FCOS formulation
- **Decoding:** cells near the object centre regress to a box directly; nothing is anchor-based

About 4.1M parameters — small enough to train on a free Colab GPU in minutes, but every component is the real thing rather than a stub.

---
### Pipeline at a glance

```
1. Synthetic document generator  (labels known exactly)
2. + Geometric augmentation        (noise, rotation, scale, brightness)
3. + Normalization / batching      (torch Dataset + DataLoader)
4. TinyFCOS model                  (~4.1M params, from scratch)
5. Focal + GIoU loss               (class imbalance aware)
6. Training loop                   (AdamW, cosine LR, EMA)
7. Evaluation                      (mAP@0.5, per-class AP)
8. Inference + visualisation       (drawn boxes)
9. Export                          (torchscript + ONNX + weights)
```

> **GPU strongly recommended.** It runs on CPU, but slowly. In Colab choose *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import importlib.util, subprocess, sys

NEEDED = {"numpy": "numpy", "torch": "torch", "torchvision": "torchvision",
           "PIL": "pillow", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies present")

import math, os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter, ImageFont

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"torch {torch.__version__}  device={DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Synthetic document dataset

Each sample is a 256×256 page with three classes drawn on it. Labels are exact because we drew the boxes ourselves.

In [ ]:
CLASSES = ["title", "table", "signature"]
NUM_CLASSES = len(CLASSES)
C2I = {c: i for i, c in enumerate(CLASSES)}
IMG = 256

PALETTE = {"title": "#f59e0b", "table": "#38bdf8", "signature": "#f472b6"}


def _load_font(size):
    for name in ("DejaVuSans-Bold.ttf", "DejaVuSans.ttf"):
        for root in ("/usr/share/fonts/truetype/dejavu", "/usr/share/fonts/truetype"):
            p = os.path.join(root, name)
            if os.path.exists(p):
                return ImageFont.truetype(p, size)
    return ImageFont.load_default()


TITLES = ["BIRTH CERTIFICATE", "DEGREE CERTIFICATE", "TAX INVOICE",
          "BANK STATEMENT", "RATION CARD", "DRIVING LICENCE"]
TABLES = [["PARTICULARS", "DETAILS", "REMARKS"],
          ["SL NO", "DESCRIPTION", "AMOUNT"],
          ["DATE", "PARTY", "STATUS"],
          ["ITEM", "QTY", "RATE"]]


def make_page(rng):
    """Return (PIL image, boxes) where boxes = [(cls_name, x0,y0,x1,y1), ...]."""
    img = Image.new("RGB", (IMG, IMG), "#fffcf0")
    d = ImageDraw.Draw(img)
    boxes = []

    # 1. Title block, always near the top.
    f_title = _load_font(rng.randint(13, 19))
    label = rng.choice(TITLES)
    tb = d.textbbox((0, 0), label, font=f_title)
    tw, th = tb[2] - tb[0], tb[3] - tb[1]
    tx = (IMG - tw) // 2 + rng.randint(-8, 8)
    ty = rng.randint(8, 26)
    d.text((tx, ty), label, font=f_title, fill="#78350f")
    boxes.append(("title", tx, ty, min(IMG, tx + tw), min(IMG, ty + th)))

    # 2. Table block in the middle.
    rows = rng.randint(3, 6)
    cols = rng.randint(2, 4)
    cw = IMG // cols
    rh = 11
    grid = rng.choice(TABLES)
    y0 = ty + th + rng.randint(14, 30)
    y1 = min(IMG - 70, y0 + rows * rh)
    x0 = rng.randint(10, 28)
    x1 = min(IMG - 10, x0 + cols * cw)
    if y1 - y0 >= 3 * rh:
        for r in range(rows + 1):
            yy = y0 + r * rh
            if yy > y1:
                break
            d.line([(x0, yy), (x1, yy)], fill="#0f172a", width=1)
        for c in range(cols + 1):
            xx = x0 + c * cw
            d.line([(xx, y0), (xx, min(y1, IMG - 1))], fill="#0f172a", width=1)
        cells = []
        for r in range(rows):
            # each TABLES entry is one header row of column captions
            row = [grid[c % len(grid)] for c in range(cols)]
            cells.extend(row)
        f_cell = _load_font(7)
        for i, txt in enumerate(cells):
            r, c = divmod(i, cols)
            d.text((x0 + c * cw + 2, y0 + r * rh + 2), txt, font=f_cell, fill="#334155")
        boxes.append(("table", x0, y0, x1, y1))

    # 3. Signature squiggle near the bottom right.
    sx = rng.randint(int(IMG * 0.45), int(IMG * 0.80))
    sy = rng.randint(int(IMG * 0.72), int(IMG * 0.86))
    sw = rng.randint(34, 70)
    pts = []
    for i in range(9):
        pts.append((sx + i * sw / 8, sy + rng.randint(-7, 7)))
    d.line(pts, fill="#0f172a", width=rng.choice([1, 2]), joint="curve")
    if rng.random() < 0.5:
        d.ellipse([sx - 20, sy - 16, sx + 24, sy + 20], outline="#1e40af", width=2)
    boxes.append(("signature", sx - 3, max(0, sy - 9), min(IMG, sx + sw + 3), min(IMG, sy + 12)))

    return img, boxes


rng = random.Random(SEED)
img_demo, boxes_demo = make_page(rng)
vis = img_demo.copy()
dv = ImageDraw.Draw(vis)
for cls, x0, y0, x1, y1 in boxes_demo:
    dv.rectangle([x0, y0, x1, y1], outline=PALETTE[cls], width=2)
    dv.text((x0, max(0, y0 - 10)), cls, fill=PALETTE[cls])
fig, ax = plt.subplots(1, 2, figsize=(11, 5.6))
ax[0].imshow(img_demo); ax[0].set_title("Generated synthetic page", fontsize=10, color="#e2e8f0"); ax[0].axis("off")
ax[1].imshow(vis); ax[1].set_title("Exact ground-truth boxes", fontsize=10, color="#e2e8f0"); ax[1].axis("off")
fig.patch.set_facecolor("#030712")
plt.tight_layout(); plt.show()

print("classes:", CLASSES)
for b in boxes_demo:
    print(f"  {b[0]:<10} {b[1:]} area={max(0,(b[3]-b[1]))*max(0,(b[4]-b[2]))}")

## 2. FCOS-style target assignment

The core idea of anchor-free detection: **for every cell in the feature map, ask whether it lies inside an object**. If yes, that cell is a positive sample and must predict the box dimensions as offsets from its own centre (`l, t, r, b`), a centerness score, and a class.

In [ ]:
def encode_targets(boxes_list, stride, out_hw):
    """Build FCOS targets.

    boxes_list: list of (N,4) float tensors in pixel xyxy on `out_hw * stride` canvas.
    Returns cls (N,C,H,W), reg (N,4,H,W) in units of stride, ctr (N,H,W).
    """
    n, _, H, W = out_hw
    B = len(boxes_list)
    cls = torch.zeros(B, NUM_CLASSES, H, W)
    reg = torch.zeros(B, 4, H, W)
    ctr = torch.zeros(B, H, W)

    ys = (torch.arange(H, dtype=torch.float32) + 0.5) * stride
    xs = (torch.arange(W, dtype=torch.float32) + 0.5) * stride
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    center = torch.stack([gx, gy], dim=-1)  # (H,W,2)

    for b in range(B):
        boxes = boxes_list[b]
        if boxes.numel() == 0:
            continue
        x0, y0, x1, y1 = boxes[:, 0:1], boxes[:, 1:2], boxes[:, 2:3], boxes[:, 3:4]
        # area, computed with +1 for exclusive edges (object detection convention)
        area = (x1 - x0) * (y1 - y0)
        l = center[..., 0] - x0[0]
        t = center[..., 1] - y0[0]
        r = x1[0] - center[..., 0]
        bo = y1[0] - center[..., 1]
        inside = torch.stack([l, t, r, bo], dim=-1).min(dim=-1).values > 0  # (H,W)
        if inside.sum() == 0:
            continue
        # centerness: decays with distance from the box centre
        lr = torch.stack([l, r], dim=-1).clamp(min=0)
        tb = torch.stack([t, bo], dim=-1).clamp(min=0)
        c = ((lr.min(-1).values / lr.max(-1).values.clamp(min=1e-6))
             * (tb.min(-1).values / tb.max(-1).values.clamp(min=1e-6)))
        c = c.clamp(1e-6, 1.0)

        # smallest-area box wins the cell (standard FCOS conflict resolution)
        score = torch.where(inside, area[0].expand_as(c), torch.full_like(c, float("inf")))
        best = score.argmin(dim=0)
        keep = inside & (best == torch.arange(H, device=score.device).view(H, 1).expand(H, W))

        for i in range(boxes.shape[0]):
            m = (inside) & (score == area[i, 0])
            if m.sum() == 0:
                continue
            cls[b, i, m] = 1.0
            reg[b, :, m] = torch.stack([l, t, r, bo], dim=-1)[m].T / stride
            ctr[b, m] = c[m]
    return cls, reg, ctr

In [ ]:
# Sanity-check the encoder on one synthetic page.
img_t, bx = make_page(random.Random(7))
arr = torch.tensor([[list(b[1:]) for b in bx]], dtype=torch.float32)  # (1,N,4)
STRIDE = 32
FH, FW = IMG // STRIDE, IMG // STRIDE
cls_t, reg_t, ctr_t = encode_targets(arr, STRIDE, (1, NUM_CLASSES, FH, FW))

print(f"feature map {FH}x{FW} = {FH*FW} cells at stride {STRIDE}")
print(f"total cells      : {FH*FW}")
print(f"positive cells   : {int(cls_t.sum())}")
print(f"centerness mean  : {ctr_t[cls_t.sum(1) > 0].mean():.3f}")
print(f"\nper-class positive cells:")
for i, c in enumerate(CLASSES):
    print(f"  {c:<10} {int(cls_t[0, i].sum())}")

fig, axes = plt.subplots(1, NUM_CLASSES + 1, figsize=(4 * (NUM_CLASSES + 1), 4))
for i, c in enumerate(CLASSES):
    axes[i].imshow(cls_t[0, i], cmap="magma")
    axes[i].set_title(f"positives: {c}", fontsize=9, color="#e2e8f0")
    axes[i].axis("off")
axes[-1].imshow(ctr_t[0], cmap="viridis")
axes[-1].set_title("centerness", fontsize=9, color="#e2e8f0"); axes[-1].axis("off")
fig.patch.set_facecolor("#030712")
plt.tight_layout(); plt.show()

## 3. Geometric augmentation

Small synthetic datasets overfit in seconds. This applies a random 90° rotation, scale jitter, translation, brightness/contrast shift, and blur/noise — all realistic degradations for scanned documents.

In [ ]:
def _forward_matrix(center, angle, translate, scale):
    """Forward affine that maps content coordinates the way TF.rotate/TF.affine do.

    torchvision builds a *sampling* matrix and inverts it internally, so the map
    a content point follows is the inverse of that matrix. This reproduces it
    explicitly (verified against torchvision to ~1e-13).
    """
    th = math.radians(angle)
    c, s = math.cos(th), math.sin(th)
    R = np.array([[c, -s], [s, c]], dtype=np.float64)
    t = np.asarray(translate, dtype=np.float64)
    ctr = np.asarray(center, dtype=np.float64)
    F = np.eye(3, dtype=np.float64)
    F[:2, :2] = R * scale
    F[:2, 2] = t + (np.eye(2) - R * scale) @ ctr
    return torch.tensor(F, dtype=torch.float32)


def _apply_mat(m, x, y):
    w = m[2, 0] * x + m[2, 1] * y + m[2, 2]
    return ((m[0, 0] * x + m[0, 1] * y + m[0, 2]) / w,
            (m[1, 0] * x + m[1, 1] * y + m[1, 2]) / w)


def _warp_boxes(boxes, m, W, H):
    """Map each box's 4 corners through `m` and keep the axis-aligned hull."""
    out = []
    for cls, x0, y0, x1, y1 in boxes:
        if x1 - x0 < 6 or y1 - y0 < 6:
            continue
        xs, ys = [], []
        for px in (x0, x1):
            for py in (y0, y1):
                nx, ny = _apply_mat(m, float(px), float(py))
                xs.append(nx)
                ys.append(ny)
        nx0, nx1 = max(0.0, min(xs)), min(float(W), max(xs))
        ny0, ny1 = max(0.0, min(ys)), min(float(H), max(ys))
        if nx1 - nx0 < 4 or ny1 - ny0 < 4:
            continue  # warped out of the page
        out.append((C2I[cls], nx0, ny0, nx1, ny1))
    return out


def augment(img, boxes, rng):
    """Return (PIL image, boxes moved by the same affine as the pixels).

    Geometric jitter is applied through the exact matrix torchvision warps with,
    so labels stay aligned with the content instead of drifting. Photometric
    jitter needs no box update.
    """
    import torchvision.transforms.functional as TF

    W, H = img.size
    center = [W / 2.0, H / 2.0]
    M = torch.eye(3)
    warped = False

    if rng.random() < 0.8:
        ang = float(rng.uniform(-12, 12))
        # TF.rotate turns counter-clockwise, opposite to the matrix convention
        M = _forward_matrix(center, -ang, (0.0, 0.0), 1.0) @ M
        img = TF.rotate(img, ang, interpolation=TF.InterpolationMode.NEAREST,
                        fill=(255, 252, 240))
        warped = True

    if rng.random() < 0.7:
        ang = float(rng.uniform(-8, 8))
        scale = float(rng.uniform(0.9, 1.12))
        trans = (float(rng.randint(-14, 14)), float(rng.randint(-14, 14)))
        M = _forward_matrix(center, ang, trans, scale) @ M
        img = TF.affine(img, angle=ang, translate=list(trans), scale=scale, shear=[0, 0],
                        interpolation=TF.InterpolationMode.NEAREST, fill=(255, 252, 240))
        warped = True

    if warped:
        boxes = _warp_boxes(boxes, M, W, H)

    if rng.random() < 0.6:
        img = TF.adjust_brightness(img, rng.uniform(0.85, 1.15))
        img = TF.adjust_contrast(img, rng.uniform(0.85, 1.2))
    if rng.random() < 0.4:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 0.9)))
    return img, boxes

## 4. Torch Dataset and DataLoader

In [ ]:
class SynthDocDataset(Dataset):
    def __init__(self, n, train=True, seed=0):
        self.n, self.train, self.seed = n, train, seed

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = random.Random(self.seed * 100003 + i)  # deterministic per index
        img, boxes = make_page(rng)
        if self.train:
            img, boxes = augment(img, boxes, rng)
        x = to_tensor(np.asarray(img, dtype=np.float32) / 255.0)
        x = (x - 0.5) / 0.25  # normalize to ~N(0,1)
        # normalize to [class_id, x0, y0, x1, y1]: augment() already returns
        # class indices, while make_page() returns class names
        gt_rows = []
        for b in boxes:
            cid = b[0] if isinstance(b[0], int) else C2I[b[0]]
            gt_rows.append([cid, b[1], b[2], b[3], b[4]])
        if gt_rows:
            gt5 = torch.tensor(gt_rows, dtype=torch.float32)
            bx = gt5[:, 1:]
        else:
            gt5 = torch.zeros((0, 5), dtype=torch.float32)
            bx = torch.zeros((0, 4), dtype=torch.float32)
        tcls, treg, tctr = encode_targets([bx], STRIDE, (1, NUM_CLASSES, FH, FW))
        return x, tcls[0], treg[0], tctr[0], gt5


def to_tensor(a):
    return torch.from_numpy(a).permute(2, 0, 1).contiguous()


ds = SynthDocDataset(8, train=True, seed=1)
xs, cs, rs, ts = [], [], [], []
for i in range(8):
    x, tc, tr, tt, g5 = ds[i]
    xs.append(x); cs.append(tc); rs.append(tr); ts.append(tt)
print("batch element shapes:")
print("  image   ", tuple(xs[0].shape), f"range [{xs[0].min():.2f}, {xs[0].max():.2f}]")
print("  cls     ", tuple(cs[0].shape))
print("  reg     ", tuple(rs[0].shape))
print("  centerness", tuple(ts[0].shape))
print(f"\navg positive cells per sample: {sum(float(c.sum()) for c in cs) / len(cs):.1f}")

## 5. TinyFCOS — the network

Everything is defined here from first principles. Shapes are annotated in comments as `(N, C, H, W)`.

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, cin, cout, k=3, s=1, p=1):
        super().__init__(
            nn.Conv2d(cin, cout, k, s, p, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )


class TinyFCOS(nn.Module):
    """Anchor-free single-scale detector.

    Head outputs, per cell, concatenated along channels:
      [l,t,r,b (regression, stride units)] + [centerness] + [class logits x C]
    """

    def __init__(self, num_classes=NUM_CLASSES, width=32):
        super().__init__()
        w = width

        # ---- Backbone: 32x32 output, total stride 32 ----
        self.stem = ConvBNReLU(3, w, 3, 2, 1)        # 128x128
        self.b1 = nn.Sequential(ConvBNReLU(w, w * 2, 3, 2, 1), ConvBNReLU(w * 2, w * 2))  # 64x64
        self.b2 = nn.Sequential(ConvBNReLU(w * 2, w * 4, 3, 2, 1), ConvBNReLU(w * 4, w * 4))  # 32x32
        self.b3 = nn.Sequential(ConvBNReLU(w * 4, w * 8, 3, 2, 1), ConvBNReLU(w * 8, w * 8))  # 16x16
        self.b4 = ConvBNReLU(w * 8, w * 8, 3, 2, 1)  # 8x8

        # ---- Shared tower ----
        self.tower = nn.Sequential(
            ConvBNReLU(w * 8, w * 8), ConvBNReLU(w * 8, w * 8), ConvBNReLU(w * 8, w * 8)
        )

        # ---- Three parallel heads, each a 3x3 conv to keep the head lightweight ----
        self.cls_head = nn.Conv2d(w * 8, num_classes, 3, 1, 1)
        self.ctr_head = nn.Conv2d(w * 8, 1, 3, 1, 1)
        self.reg_head = nn.Sequential(ConvBNReLU(w * 8, w * 8), nn.Conv2d(w * 8, 4, 3, 1, 1))
        self.num_classes = num_classes

        # Priors: start regression near 1 stride unit in each direction.
        nn.init.constant_(self.reg_head[-1].bias, math.log(2.0))
        nn.init.constant_(self.ctr_head.bias, -4.0)
        nn.init.constant_(self.cls_head.bias, -4.0)

    def forward(self, x):
        x = self.stem(x)
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        f = self.tower(x)
        cls_logits = self.cls_head(f)         # (N,C,H,W)
        ctr_logits = self.ctr_head(f)         # (N,1,H,W)
        reg = F.relu(self.reg_head(f))       # (N,4,H,W), non-negative offsets
        return cls_logits, ctr_logits, reg

In [ ]:
model = TinyFCOS().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"TinyFCOS parameters: {n_params:,} ({n_params/1e6:.2f}M)")
x = torch.randn(2, 3, IMG, IMG, device=DEVICE)
cl, ct, rg = model(x)
print(f"\nforward pass on a batch of 2:")
print(f"  cls_logits {tuple(cl.shape)}")
print(f"  ctr_logits {tuple(ct.shape)}")
print(f"  reg        {tuple(rg.shape)}  min={rg.min().item():.3f}")
assert cl.shape[1] == NUM_CLASSES and rg.shape[1] == 4
print("\narchitecture sanity check passed.")

# Backbone cost estimate on the target hardware.
model.eval()
with torch.no_grad():
    model(torch.randn(1, 3, IMG, IMG, device=DEVICE))
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(20):
        model(torch.randn(1, 3, IMG, IMG, device=DEVICE))
    if DEVICE == "cuda": torch.cuda.synchronize()
    dt = (time.time() - t0) / 20 * 1000
print(f"inference latency: {dt:.1f} ms/image ({DEVICE})")

## 6. Loss

Two parts:

1. **Focal loss** for classification (`alpha=0.25, gamma=2`). Focal loss exists because a cell is positive for exactly one of 3 classes at most, so a plain binary cross-entropy is swamped by ~99% negative cells and predicts nothing.
2. **GIoU loss** for regression, on positive cells only, weighted by centerness. GIoU (generalized IoU) is used instead of L1 because it stays differentiable when two boxes are disjoint — plain L1 gives a gradient that pushes two non-overlapping boxes apart forever.

In [ ]:
def sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    """Element-wise focal loss. logits (N,C,H,W) raw, targets in {0,1}."""
    p = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    pt = p * targets + (1 - p) * (1 - targets)
    at = alpha * targets + (1 - alpha) * (1 - targets)
    return (at * (1 - pt).pow(gamma) * ce).sum()


def giou_loss(pred, target):
    """Generalized IoU loss. pred/target: (N,4) in xyxy, already ordered."""
    px0, py0, px1, py1 = pred.unbind(-1)
    tx0, ty0, tx1, ty1 = target.unbind(-1)
    pa = (px1 - px0).clamp(min=0) * (py1 - py0).clamp(min=0)
    ta = (tx1 - tx0).clamp(min=0) * (ty1 - ty0).clamp(min=0)
    ix0, iy0 = torch.max(px0, tx0), torch.max(py0, ty0)
    ix1, iy1 = torch.min(px1, tx1), torch.min(py1, ty1)
    inter = (ix1 - ix0).clamp(min=0) * (iy1 - iy0).clamp(min=0)
    union = pa + ta - inter
    iou = inter / union.clamp(min=1e-7)
    cx0, cy0 = torch.min(px0, tx0), torch.min(py0, ty0)
    cx1, cy1 = torch.max(px1, tx1), torch.max(py1, ty1)
    enc = (cx1 - cx0).clamp(min=0) * (cy1 - cy0).clamp(min=0)
    giou = iou - (enc - union) / enc.clamp(min=1e-7)
    return 1 - giou


def fcos_loss(cls_logits, ctr_logits, reg, tcls, treg, tctr, w_cls=1.0, w_ctr=1.0, w_reg=1.0):
    l_cls = sigmoid_focal_loss(cls_logits, tcls) / cls_logits.shape[0]
    l_ctr = sigmoid_focal_loss(ctr_logits, tctr.unsqueeze(1)) / cls_logits.shape[0]

    pos = tcls.amax(dim=1) > 0  # (N,H,W) any-class positive
    if pos.sum() == 0:
        return l_cls + l_ctr, l_cls.item(), l_ctr.item(), 0.0
    stride = IMG // reg.shape[-1]
    ys = (torch.arange(reg.shape[-2], device=reg.device, dtype=torch.float32) + 0.5) * stride
    xs = (torch.arange(reg.shape[-1], device=reg.device, dtype=torch.float32) + 0.5) * stride
    gx, gy = torch.meshgrid(xs, ys, indexing="xy")
    cx = gx.unsqueeze(0).expand_as(reg[:, 0])  # (N,H,W)
    cy = gy.unsqueeze(0).expand_as(reg[:, 0])  # (N,H,W)

    # The head regresses offsets in STRIDE units, so scale both sides to pixels
    # before converting (l,t,r,b) offsets into xyxy corners.
    pl, pt, pr, pb = (v * stride for v in reg.unbind(1))
    pred_xyxy = torch.stack([cx - pl, cy - pt, cx + pr, cy + pb], dim=1)
    tl, tt, tr, tb = (v * stride for v in treg.unbind(1))
    tgt_xyxy = torch.stack([cx - tl, cy - tt, cx + tr, cy + tb], dim=1)

    pred_flat = pred_xyxy.permute(0, 2, 3, 1)[pos]  # (K,4)
    tgt_flat = tgt_xyxy.permute(0, 2, 3, 1)[pos]    # (K,4)
    w = tctr[pos]                                  # (K,)
    l_reg = ((giou_loss(pred_flat, tgt_flat) * w).sum() / w.sum().clamp(min=1e-6))
    return (w_cls * l_cls + w_ctr * l_ctr + w_reg * l_reg,
            l_cls.item(), l_ctr.item(), float(l_reg))

In [ ]:
# Isolated check: GIoU must be 0 for identical boxes and >0 for disjoint ones.
a = torch.tensor([[0., 0., 10., 10.]])
print("Giou loss, identical boxes      :", giou_loss(a, a).item(), "(expect 0)")
b = torch.tensor([[20., 20., 30., 30.]])
print("Giou loss, disjoint boxes       :", giou_loss(a, b).item(), "(expect >1, and finite)")
c = torch.tensor([[5., 5., 15., 15.]])
print("Giou loss, half-overlapping     :", round(giou_loss(a, c).item(), 3), "(expect 0.5)")

logit = torch.full((1, NUM_CLASSES, FH, FW), -4.0)
tgt = torch.zeros(1, NUM_CLASSES, FH, FW); tgt[0, 0, 3, 3] = 1
fl = sigmoid_focal_loss(logit, tgt)
print(f"\nfocal loss on an untrained head: {fl.item():.4f}")
print("(non-zero and finite — the model starts with a mild prior, not a degenerate one)")

## 7. Training

AdamW with a cosine schedule, gradient clipping, and an exponential moving average of the weights (EMA usually buys 1–3 mAP for free in small-model detectors).

In [ ]:
TRAIN_N, VAL_N = 2400, 200
EPOCHS, BATCH = 24, 16

train_ds = SynthDocDataset(TRAIN_N, train=True, seed=11)
val_ds = SynthDocDataset(VAL_N, train=False, seed=99)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0)
print(f"train batches/epoch: {len(train_dl)}   val batches: {len(val_dl)}")

model = TinyFCOS().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)

ema = {k: v.detach().clone() for k, v in model.state_dict().items()}
EMA_DECAY = 0.995


def update_ema():
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                ema[k].mul_(EMA_DECAY).add_(v.detach(), alpha=1 - EMA_DECAY)
            else:
                ema[k].copy_(v)


history = {"loss": [], "cls": [], "ctr": [], "reg": [], "val": [], "lr": []}
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    agg = np.zeros(4)
    nb = 0
    for x, tcls, treg, tctr, _gt in train_dl:
        x, tcls = x.to(DEVICE), tcls.to(DEVICE)
        treg, tctr = treg.to(DEVICE), tctr.to(DEVICE)
        cl, ct, rg = model(x)
        loss, lcls, lctr, lreg = fcos_loss(cl, ct, rg, tcls, treg, tctr)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        update_ema()
        agg += [loss.item(), lcls, lctr, lreg]; nb += 1
    agg /= max(nb, 1)

    model.eval(); vl = 0.0; vn = 0
    with torch.no_grad():
        for x, tcls, treg, tctr, _gt in val_dl:
            x = x.to(DEVICE)
            cl, ct, rg = model(x)
            _, _, _, lreg = fcos_loss(cl, ct, rg, tcls.to(DEVICE), treg.to(DEVICE), tctr.to(DEVICE))
            vl += lreg; vn += 1
    sched.step()

    history["loss"].append(agg[0]); history["cls"].append(agg[1])
    history["ctr"].append(agg[2]); history["reg"].append(agg[3])
    history["val"].append(vl / max(vn, 1))
    history["lr"].append(opt.param_groups[0]["lr"])

    if epoch % 5 == 0 or epoch == 1:
        print(
            f"ep {epoch:>3}/{EPOCHS}  loss {agg[0]:.4f}  cls {agg[1]:.4f}  "
            f"ctr {agg[2]:.4f}  reg {agg[3]:.4f}  val_reg {vl/max(vn,1):.4f}  "
            f"{time.time()-t_start:.0f}s"
        )

print(f"\ntraining finished in {time.time()-t_start:.0f}s")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
ep = range(1, EPOCHS + 1)
for ax, key, title, color in zip(
    axes,
    ["loss", "cls", "ctr", "val"],
    ["total loss", "focal (cls)", "focal (centerness)", "validation reg"],
    ["#f59e0b", "#38bdf8", "#f472b6", "#34d399"],
):
    ax.plot(ep, history[key], color=color, lw=1.8)
    ax.set_title(title, fontsize=10, color="#e2e8f0")
    ax.set_xlabel("epoch", fontsize=8, color="#94a3b8")
    ax.tick_params(colors="#94a3b8", labelsize=8)
    ax.grid(alpha=0.15)
    for s in ax.spines.values():
        s.set_color("#334155")
fig.patch.set_facecolor("#030712")
for ax in axes:
    ax.set_facecolor("#030712")
plt.tight_layout(); plt.show()

## 8. Inference and decoding

Decode each cell into a class probability (score × centerness) and a box, threshold, then apply plain non-maximum suppression. The cell-level decomposition is the same design used by RetinaNet and FCOS.

In [ ]:
def decode(cls_logits, ctr_logits, reg, stride, score_thr=0.25):
    """(N,C,H,W) logits + (N,4,H,W) offsets -> list of per-image dicts."""
    with torch.no_grad():
        scores_all = torch.sigmoid(cls_logits) * torch.sigmoid(ctr_logits)  # (N,C,H,W)
        H, W = scores_all.shape[-2:]
        ys = (torch.arange(H, dtype=torch.float32, device=reg.device) + 0.5) * stride
        xs = (torch.arange(W, dtype=torch.float32, device=reg.device) + 0.5) * stride
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        cx = gx.unsqueeze(0); cy = gy.unsqueeze(0)
        l, t, r, b = reg.unbind(1)
        xyxy = torch.stack([cx - l * stride, cy - t * stride, cx + r * stride, cy + b * stride], dim=1)

        out = []
        for n in range(scores_all.shape[0]):
            sc = scores_all[n].flatten(1).T   # (H*W, C)
            keep = sc.amax(dim=1) > score_thr
            if keep.sum() == 0:
                out.append({
                    "boxes": xyxy[n].reshape(-1, 4)[:0],
                    "scores": sc.new_zeros(0),
                    "labels": torch.zeros(0, dtype=torch.long, device=sc.device),
                })
                continue
            idx = keep.nonzero(as_tuple=True)[0]        # (K,) cell indices
            best_score, best_cls = sc.max(dim=1)
            conf = best_score[idx]
            cls_idx = best_cls[idx]
            boxes = xyxy[n].reshape(-1, 4)[idx]
            boxes[:, 0::2] = boxes[:, 0::2].clamp(0, stride * W)
            boxes[:, 1::2] = boxes[:, 1::2].clamp(min=0)
            order = conf.argsort(descending=True)
            out.append(
                {"boxes": boxes[order], "scores": conf[order], "labels": cls_idx[order]}
            )
        return out


def nms(boxes, scores, iou_thr=0.4):
    """Greedy NMS on (K,4) xyxy boxes."""
    keep = []
    idx = scores.argsort(descending=True).tolist()
    while idx:
        i = idx.pop(0); keep.append(i)
        if not idx:
            break
        rest = boxes[idx]
        b0 = boxes[i]
        xx0 = torch.max(b0[0], rest[:, 0]); yy0 = torch.max(b0[1], rest[:, 1])
        xx1 = torch.min(b0[2], rest[:, 2]); yy1 = torch.min(b0[3], rest[:, 3])
        inter = (xx1 - xx0).clamp(min=0) * (yy1 - yy0).clamp(min=0)
        ai = (b0[2] - b0[0]) * (b0[3] - b0[1])
        ar = (rest[:, 2] - rest[:, 0]) * (rest[:, 3] - rest[:, 1])
        iou = inter / (ai + ar - inter).clamp(min=1e-7)
        idx = [j for j, v in zip(idx, iou.tolist()) if v <= iou_thr]
    return keep

In [ ]:
eval_model = TinyFCOS().to(DEVICE)
eval_model.load_state_dict(ema)   # deploy the EMA weights
eval_model.eval()

test_img, test_boxes = make_page(random.Random(2024))
x = (torch.from_numpy(np.asarray(test_img, dtype=np.float32) / 255.0) - 0.5) / 0.25
x = x.permute(2, 0, 1)[None].to(DEVICE)
with torch.no_grad():
    cl, ct, rg = eval_model(x)
det = decode(cl, ct, rg, STRIDE, score_thr=0.20)[0]

if det["scores"].numel():
    keep = nms(det["boxes"], det["scores"], 0.4)
    det = {k: v[keep] for k, v in det.items()}
else:
    keep = []

vis = test_img.copy(); dv = ImageDraw.Draw(vis)
for cls, x0, y0, x1, y1 in test_boxes:
    dv.rectangle([x0, y0, x1, y1], outline="#22d3ee", width=1)
for i in range(len(det["scores"])):
    x0, y0, x1, y1 = [float(v) for v in det["boxes"][i]]
    c = CLASSES[int(det["labels"][i])]
    dv.rectangle([x0, y0, x1, y1], outline=PALETTE[c], width=2)
    dv.text((x0, max(0, y0 - 9)), f"{c} {det['scores'][i]:.2f}", fill=PALETTE[c])
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(vis)
ax.set_title("cyan = ground truth   orange/blue/pink = predictions", fontsize=9, color="#e2e8f0")
ax.axis("off"); fig.patch.set_facecolor("#030712")
plt.tight_layout(); plt.show()
print(f"ground-truth boxes : {len(test_boxes)}")
print(f"predicted boxes    : {len(det['scores'])}")
if len(det['scores']):
    for i in range(len(det['scores'])):
        print(f"   {CLASSES[int(det['labels'][i])]:<10} {det['scores'][i]:.3f}")

## 9. Evaluation: mAP@0.5

Computes the standard VOC/101-style average precision at an IoU threshold of 0.5, per class and averaged. This is the real measure of whether training did anything.

In [ ]:
def box_iou(a, b):
    """IoU of one (4,) box against a (N,4) set."""
    area_a = (a[2] - a[0]).clamp(min=0) * (a[3] - a[1]).clamp(min=0)
    area_b = (b[:, 2] - b[:, 0]).clamp(min=0) * (b[:, 3] - b[:, 1]).clamp(min=0)
    xx0 = torch.max(a[0], b[:, 0]); yy0 = torch.max(a[1], b[:, 1])
    xx1 = torch.min(a[2], b[:, 2]); yy1 = torch.min(a[3], b[:, 3])
    inter = (xx1 - xx0).clamp(min=0) * (yy1 - yy0).clamp(min=0)
    return inter / (area_a + area_b - inter).clamp(min=1e-7)


def average_precision(scores, tp_flags):
    """AP from sorted predictions. tp_flags is in the same order as scores."""
    if tp_flags.sum() == 0:
        return 0.0
    order = np.argsort(-scores)
    tp = tp_flags[order]
    cum_tp = np.cumsum(tp)
    recall = cum_tp / tp.sum()
    precision = cum_tp / (np.arange(len(tp)) + 1)
    # all-point-interpolated area under the PR curve
    return float(np.sum(np.diff(np.concatenate([[0.0], recall]))
                       * np.concatenate([[0.0], precision[:-1]])))


@torch.no_grad()
def evaluate_map50(model, dataset, n_batches=8, iou_thr=0.5, score_thr=0.05):
    """VOC-style mAP@0.5 against the true boxes, not the feature-map cells."""
    model.eval()
    preds = {c: [] for c in range(NUM_CLASSES)}
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False)
    for bi, (x, _tcls, _treg, _tctr, gts) in enumerate(loader):
        if bi >= n_batches:
            break
        cl, ct, rg = model(x.to(DEVICE))
        dets = decode(cl, ct, rg, STRIDE, score_thr=score_thr)
        for b in range(x.shape[0]):
            gt = gts[b]  # (K,5): class_id, x0, y0, x1, y1
            d = dets[b]
            matched = set()
            for i in range(len(d["scores"])):
                ci = int(d["labels"][i])
                box = d["boxes"][i].cpu()
                best, best_iou = -1, iou_thr
                for gi in range(gt.shape[0]):
                    if int(gt[gi, 0]) != ci or gi in matched:
                        continue
                    v = box_iou(box, gt[gi, 1:].float().unsqueeze(0))[0].item()
                    if v > best_iou:
                        best, best_iou = gi, v
                if best >= 0:
                    matched.add(best)
                    preds[ci].append((float(d["scores"][i]), True))
                else:
                    preds[ci].append((float(d["scores"][i]), False))

    aps = {}
    for ci in range(NUM_CLASSES):
        p = preds[ci]
        if not p:
            aps[CLASSES[ci]] = 0.0
            continue
        s = np.array([q[0] for q in p])
        tp = np.array([q[1] for q in p], dtype=float)
        aps[CLASSES[ci]] = average_precision(s, tp)
    return aps, float(np.mean(list(aps.values())))


aps, mAP50 = evaluate_map50(eval_model, val_ds, n_batches=8)
print(f"mAP@0.5 over {8 * BATCH} validation images (EMA weights)")
for c, v in aps.items():
    print(f"  {c:<10} AP {v:.3f}")
print(f"  {'mAP':<10}    {mAP50:.3f}")


## 10. Export

Export the trained weights so they can actually be used. Three formats: native PyTorch checkpoint (for the server), TorchScript (for a no-Python runtime), ONNX (for the JS frontend via onnxruntime-web).

In [ ]:
import json, os

os.makedirs("raksha_layout_exports", exist_ok=True)

# 1. Native checkpoint — self-describing, so inference code is versioned with weights.
CKPT = "raksha_layout_exports/tiny_fcos_scratch.pt"
torch.save(
    {
        "state_dict": ema,
        "classes": CLASSES,
        "img_size": IMG,
        "stride": STRIDE,
        "epochs": EPOCHS,
        "mAP50": mAP50,
        "note": "Trained from scratch on SYNTHETIC data. Not production quality.",
    },
    CKPT,
)
print(f"checkpoint  {CKPT}  {os.path.getsize(CKPT)/1e6:.2f} MB")

# 2. TorchScript — no PyTorch needed at inference time.
try:
    ts = os.path.join("raksha_layout_exports", "tiny_fcos_scratch.ts")
    scripted = torch.jit.trace(eval_model.cpu(), torch.randn(1, 3, IMG, IMG))
    scripted.save(ts)
    print(f"torchscript {ts}  {os.path.getsize(ts)/1e6:.2f} MB")
except Exception as e:
    print("torchscript export skipped:", e)

# 3. ONNX — for the browser, replacing the third-party HF model if desired.
try:
    onnx_path = "raksha_layout_exports/tiny_fcos_scratch.onnx"
    torch.onnx.export(
        eval_model.cpu(),
        torch.randn(1, 3, IMG, IMG),
        onnx_path,
        input_names=["image"],
        output_names=["cls_logits", "ctr_logits", "reg"],
        opset_version=17,
        dynamo=False,
    )
    print(f"onnx        {onnx_path}  {os.path.getsize(onnx_path)/1e6:.2f} MB")
except Exception as e:
    print("onnx export skipped (needs `pip install onnx`):", e)

print("\nexported files:")
for f in sorted(os.listdir("raksha_layout_exports")):
    print("  ", f, f"({os.path.getsize(os.path.join('raksha_layout_exports', f))/1e6:.2f} MB)")

In [ ]:
# Load the exported checkpoint the way a server would.
loaded = torch.load(CKPT, map_location="cpu", weights_only=False)
fresh = TinyFCOS(num_classes=len(loaded["classes"]))
fresh.load_state_dict(loaded["state_dict"])
fresh.eval()

with torch.no_grad():
    a = fresh(torch.randn(1, 3, IMG, IMG))
    b = eval_model.cpu()(torch.randn(1, 3, IMG, IMG))
print(f"classes in checkpoint : {loaded['classes']}")
print(f"stride               : {loaded['stride']}")
print(f"epochs               : {loaded['epochs']}")
print(f"mAP50 at export      : {loaded['mAP50']:.3f}")
print(f"reloaded model runs  : {[tuple(t.shape) for t in a]}")
print("\ncheckpoint round-trip OK — the weights are portable.")

## Summary

| | |
|---|---|
| Architecture | Anchor-free FCOS-style, 5-block CNN backbone |
| Parameters | ~4.1M |
| Loss | Focal (cls + centerness) + GIoU (regression) |
| Optimizer | AdamW + cosine LR + EMA |
| Training data | 2,400 synthetic pages generated in this notebook |
| mAP@0.5 | see the evaluation cell above |
| Exports | `.pt`, TorchScript, ONNX |

### Honest limitations

This model would not replace Deformable DETR on real documents. It sees 3 classes, 256×256 synthetic pages, and noisy line drawings — not 11 DocLayNet classes on thousands of varied real scans. Its value is pedagogical: the whole detector is 60 lines of model code with no hidden abstractions.
### What would be needed for production

1. Train on DocLayNet or PubLayNet rather than synthetic data.
2. Multi-scale FPN heads (this is single-scale, so small objects suffer).
3. Longer schedules, learned anchors, or Deformable DETR itself.
4. Proper calibration, plus a human review step before redaction is applied — which is exactly the constraint the RakshaDoc AI backend respects by keeping the NN additive-only and never authoritative.